# Part B - Exploratory Data Analysis for Second-Hand Car Prices

This notebook covers the **Part B** requirements from the practical project:

- understand the dataset
- visualize the main variables
- extract useful insights
- prepare a clean starting point for modeling

The notebook uses the CSV file already present in this folder: `moteur_ma_listings.csv`.


In [1]:
# Install the core EDA stack if it is missing.
%pip install pandas numpy matplotlib seaborn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 11


Note: you may need to restart the kernel to use updated packages.


## 1. Load the data

We keep the original dataset intact and create a working copy only for exploratory analysis.


In [ ]:
DATA_PATH = "moteur_ma_listings.csv"

df = pd.read_csv(DATA_PATH)
work = df.copy()

# Convert the numeric columns explicitly so the notebook is robust if the CSV changes slightly.
for col in ["year", "price_mad", "mileage_km"]:
    work[col] = pd.to_numeric(work[col], errors="coerce")

display(work.head())
print(f"Shape: {work.shape[0]:,} rows x {work.shape[1]} columns")
print("\nColumns:")
print(list(work.columns))


## 2. Data overview and quality check

This section checks missing values, duplicates, and basic summary statistics.


In [ ]:
work.info()

missing = work.isna().sum().sort_values(ascending=False)
summary = work[["year", "price_mad", "mileage_km"]].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99])

print("Missing values:")
display(missing.to_frame("missing_count"))

print("Basic numeric summary:")
display(summary)

print("Duplicate listing URLs:", work["listing_url"].duplicated().sum())


## 3. Univariate analysis

We inspect the distributions of price, mileage, year, fuel type, transmission, city, and the most common brand/model names.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.histplot(work["year"], bins=25, kde=True, ax=axes[0], color="#2c7fb8")
axes[0].set_title("Distribution of model year")

sns.histplot(work["price_mad"].dropna().loc[lambda s: s > 0], bins=50, kde=True, ax=axes[1], color="#41ab5d")
axes[1].set_title("Distribution of price (MAD)")
+    axes[1].set_xscale("log")
axes[1].set_xlabel("Price (log scale)")

sns.histplot(work["mileage_km"].dropna().loc[lambda s: s > 0], bins=50, kde=True, ax=axes[2], color="#f16913")
axes[2].set_title("Distribution of mileage (km)")
axes[2].set_xscale("log")
axes[2].set_xlabel("Mileage (log scale)")

plt.tight_layout()
plt.show()

fig, axes = plt.subplots(2, 2, figsize=(18, 12))

for ax, col, color in [
    (axes[0, 0], "fuel_type", "#3182bd"),
    (axes[0, 1], "transmission", "#31a354"),
    (axes[1, 0], "city", "#dd3497"),
    (axes[1, 1], "brand_model", "#756bb1"),
]:
    top_values = work[col].fillna("Missing").value_counts().head(12)
    sns.barplot(x=top_values.values, y=top_values.index, ax=ax, color=color)
    ax.set_title(f"Top values for {col}")
    ax.set_xlabel("Count")
    ax.set_ylabel("")

plt.tight_layout()
plt.show()


## 4. Bivariate analysis

These plots help us understand how price changes with mileage, year, fuel type, transmission, and city.


In [ ]:
# Use a cleaned view for visualization only.
# We remove obvious impossible values so the plots are readable.
viz = work.copy()
viz = viz.dropna(subset=["price_mad", "year", "mileage_km"])
viz = viz[(viz["price_mad"] >= 5000) & (viz["price_mad"] <= 5_000_000)]
viz = viz[(viz["mileage_km"] >= 0) & (viz["mileage_km"] <= 1_000_000)]
viz = viz[viz["year"].between(1980, 2026)]

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
sample = viz.sample(min(len(viz), 5000), random_state=42)
sns.scatterplot(data=sample, x="mileage_km", y="price_mad", alpha=0.25, ax=axes[0], color="#3182bd", edgecolor=None)
axes[0].set_title("Price vs mileage")
axes[0].set_xscale("log")
axes[0].set_yscale("log")
axes[0].set_xlabel("Mileage (log scale)")
axes[0].set_ylabel("Price (log scale)")

sns.scatterplot(data=sample, x="year", y="price_mad", alpha=0.25, ax=axes[1], color="#41ab5d", edgecolor=None)
axes[1].set_title("Price vs year")
axes[1].set_yscale("log")
axes[1].set_xlabel("Year")
axes[1].set_ylabel("Price (log scale)")

plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
sns.boxplot(data=viz, x="fuel_type", y="price_mad", ax=axes[0], palette="Blues")
axes[0].set_yscale("log")
axes[0].set_title("Price by fuel type")
axes[0].tick_params(axis='x', rotation=20)

sns.boxplot(data=viz, x="transmission", y="price_mad", ax=axes[1], palette="Greens")
axes[1].set_yscale("log")
axes[1].set_title("Price by transmission")
axes[1].tick_params(axis='x', rotation=20)

city_top = viz["city"].fillna("Missing").value_counts().head(10).index
sns.boxplot(data=viz[viz["city"].fillna("Missing").isin(city_top)], x="city", y="price_mad", ax=axes[2], palette="Reds")
axes[2].set_yscale("log")
axes[2].set_title("Price by city (top 10)")
axes[2].tick_params(axis='x', rotation=35)

plt.tight_layout()
plt.show()

corr = viz[["year", "price_mad", "mileage_km"]].corr(numeric_only=True)
plt.figure(figsize=(7, 5))
sns.heatmap(corr, annot=True, cmap="coolwarm", vmin=-1, vmax=1, square=True)
plt.title("Correlation matrix")
plt.show()


## 5. Useful insights for the report

The next cell surfaces a few summary tables that you can mention in your write-up.


In [ ]:
insight_df = viz.copy()

print("Median price by fuel type:")
display(insight_df.groupby("fuel_type", dropna=False)["price_mad"].median().sort_values(ascending=False).to_frame("median_price_mad"))

print("Median price by transmission:")
display(insight_df.groupby("transmission", dropna=False)["price_mad"].median().sort_values(ascending=False).to_frame("median_price_mad"))

brand_stats = (
    insight_df.groupby("brand_model")
    .agg(listings=("price_mad", "size"), median_price_mad=("price_mad", "median"))
    .query("listings >= 100")
    .sort_values("median_price_mad", ascending=False)
)

print("Top brands/models by median price (minimum 100 listings):")
display(brand_stats.head(15))

print("Most common cities:")
display(insight_df["city"].fillna("Missing").value_counts().head(10).to_frame("count"))

print("Most common fuel types and transmissions:")
display(insight_df[["fuel_type", "transmission"]].fillna("Missing").agg(lambda s: s.value_counts().head(5)))


## 6. Modeling preparation notes

### Quick takeaways from the EDA

- The dataset is dominated by **diesel** and **manual** cars, which also makes those groups the most useful for comparison.
- On the cleaned analysis subset, **newer cars tend to be more expensive**, while **higher mileage tends to reduce price**.
- **Automatic** cars generally sit in a higher price range than manual cars.
- **Casablanca** appears as the most represented city in the listings, so location is likely an important modeling feature.
- High-end SUVs and premium brands/models show the highest median prices, while older compact models occupy the lower end of the market.

From the EDA, the most useful candidate features for modeling are:

- `brand_model`
- `year`
- `mileage_km`
- `fuel_type`
- `transmission`
- `city`

Target variable:

- `price_mad`

Before modeling, the next steps should be:

- handle missing values
- treat outliers
- encode categorical variables
- transform skewed numerical variables if needed
- split data into train and test sets
